# 📏 Phase 2: Baseline Evaluation (Base Model — BEFORE Fine-Tuning)

**THIS IS THE MOST IMPORTANT STEP.**

We run the raw Qwen2.5-7B-Instruct model on our test set WITH the best possible prompt.
These numbers become Row 1 of your 3-way comparison table.

Metrics we measure:
1. **JSON Validity Rate** — % of outputs that are parseable valid JSON
2. **Exact Match Accuracy** — % of outputs that exactly match ground truth JSON
3. **Field Coverage Rate** — % of required fields present and correctly populated
4. **Refusal Correctness** — does the model correctly handle ambiguous inputs

In [ ]:
# ── Global config ─────────────────────────────────────────────────────────────
BASE_MODEL  = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN = 1024
SEED        = 42
import os, json
os.environ["HF_TOKEN"] = "hf_XXXXXXXXXXXXXXXXXXXXXXXX"  # ← paste token

In [ ]:
# ── Load model in 4-bit (saves VRAM, fine for eval) ───────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print('✅ Base model loaded')

In [ ]:
# ── Define evaluation metrics ─────────────────────────────────────────────────
import json

def is_valid_json(text):
    """Check if output is parseable JSON."""
    try:
        json.loads(text.strip())
        return True
    except:
        return False

def exact_match(pred, ground_truth):
    """Check if parsed JSON exactly matches ground truth."""
    try:
        pred_obj = json.loads(pred.strip())
        gt_obj   = json.loads(ground_truth.strip())
        return pred_obj == gt_obj
    except:
        return False

def field_coverage(pred, ground_truth):
    """What % of required fields are present AND correct."""
    try:
        pred_obj = json.loads(pred.strip())
        gt_obj   = json.loads(ground_truth.strip())
        correct = sum(1 for k, v in gt_obj.items() if pred_obj.get(k) == v)
        return correct / len(gt_obj)
    except:
        return 0.0

print('✅ Metric functions defined')

In [ ]:
# ── Inference function with best-effort prompt engineering ────────────────────
SYSTEM_PROMPT = """You are a precise JSON extraction assistant. 
Given unstructured text, extract the requested fields and return ONLY valid JSON.
Do not include any explanation, markdown, or extra text. Output only the JSON object."""

def run_inference(instruction, input_text, model, tokenizer, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{instruction}\n\nText: {input_text}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,   # Low temp for deterministic JSON output
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

print('✅ Inference function ready')

In [ ]:
# ── Run evaluation on test set ────────────────────────────────────────────────
# Using 100 examples for speed (full test ~200 examples)
from tqdm import tqdm

with open('./data/test_sft.json') as f:
    test_data = json.load(f)

EVAL_SAMPLE = test_data[:100]  # evaluate on 100 examples

results = []
for ex in tqdm(EVAL_SAMPLE, desc="Evaluating base model"):
    pred = run_inference(ex['instruction'], ex['input_text'], model, tokenizer)
    results.append({
        'schema': ex['schema'],
        'input': ex['input_text'],
        'ground_truth': ex['output'],
        'prediction': pred,
        'valid_json': is_valid_json(pred),
        'exact_match': exact_match(pred, ex['output']),
        'field_coverage': field_coverage(pred, ex['output']),
    })

# Aggregate metrics
n = len(results)
baseline_metrics = {
    'model': 'Base Model (Qwen2.5-7B-Instruct)',
    'json_validity_rate': sum(r['valid_json'] for r in results) / n,
    'exact_match_accuracy': sum(r['exact_match'] for r in results) / n,
    'avg_field_coverage': sum(r['field_coverage'] for r in results) / n,
    'n_samples': n
}

print('\n' + '='*60)
print('📊 BASELINE METRICS (Base Model — Before Fine-Tuning)')
print('='*60)
for k, v in baseline_metrics.items():
    if isinstance(v, float):
        print(f'  {k:30s}: {v:.1%}')
    else:
        print(f'  {k:30s}: {v}')

# Save for comparison later
with open('./data/baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)
print('\n✅ Saved to baseline_metrics.json — DO NOT LOSE THIS FILE')

In [ ]:
# ── Qualitative check: look at 5 examples ────────────────────────────────────
print('=== SAMPLE PREDICTIONS (Base Model) ===')
for r in results[:5]:
    print(f"Schema    : {r['schema']}")
    print(f"Input     : {r['input'][:80]}...")
    print(f"Expected  : {r['ground_truth']}")
    print(f"Got       : {r['prediction']}")
    print(f"Valid JSON: {r['valid_json']} | Exact: {r['exact_match']}")
    print('-' * 60)